In [18]:
import pandas as pd
import numpy as np


In [19]:
# Training data
y_train = pd.read_csv("original/water_quality_training_dataset.csv")
x_train_All = pd.read_csv("Data/training_merged.csv")
print("Training Response Features Shape: ", y_train.shape)
print("Training Features Shape: ", x_train_All.shape)
landsat_only_train = pd.read_csv("original/landsat_features_training.csv")
print(landsat_only_train.shape)

# Validation data
submission_template = pd.read_csv("Data/submission_template.csv")
x_test_All = pd.read_csv("Data/validation_merged.csv")


print(landsat_only_train.head())

Training Response Features Shape:  (9319, 6)
Training Features Shape:  (8985, 88)
(9319, 9)
    Latitude  Longitude Sample Date      nir    green   swir16   swir22  \
0 -28.760833  17.730278  02-01-2011  11190.0  11426.0   7687.5   7645.0   
1 -26.861111  28.884722  03-01-2011  17658.5   9550.0  13746.5  10574.0   
2 -26.450000  28.085833  03-01-2011  15210.0  10720.0  17974.0  14201.0   
3 -27.671111  27.236944  03-01-2011  14887.0  10943.0  13522.0  11403.0   
4 -27.356667  27.286389  03-01-2011  16828.5   9502.5  12665.5   9643.0   

       NDMI     MNDWI  
0  0.185538  0.195595  
1  0.124566 -0.180134  
2 -0.083293 -0.252805  
3  0.048048 -0.105416  
4  0.141147 -0.142683  


In [20]:
test_concat = y_train.merge(
    landsat_only_train.reset_index(), 
    on=["Latitude", "Longitude"], 
    how="left"
)

In [21]:
print(y_train.shape)
y_train[["Longitude","Latitude"]].nunique()

(9319, 6)


Longitude    161
Latitude     161
dtype: int64

In [22]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'.')

from panel_model_v2 import (
    PanelWaterQualityModel,
    spatial_cv,
    compute_site_means,
    TIME_VARIANT_COLS,
    TARGETS
)
print("imports OK")

imports OK


In [23]:
train = pd.read_csv("Data/training_merged.csv")
val = pd.read_csv("Data/validation_merged.csv")
submission_template = pd.read_csv("Data/submission_template.csv")

print(f"Train: {train.shape} | {train.groupby(["Latitude","Longitude"]).ngroups} sites")
print(f"Validation: {val.shape} | {val.groupby(["Latitude","Longitude"]).ngroups} sites")

Train: (8985, 88) | 162 sites
Validation: (200, 85) | 24 sites


In [24]:
def add_geology_flags(df):
    def flags(name):
        n = str(name).lower()
        is_karoo = int(any(k in n for k in [
            'karoo','beaufort','ecca','dwyka','molteno','elliot','clarens',
            'vryheid','volksrust','tarkastad','fort brown','waterford',
            'prince albert','emakwezini','pietermaritzburg','dolerite',
        ]))
        is_cape = int(any(k in n for k in [
            'cape supergroup','witteberg','bokkeveld','table mountain',
            'nardouw','ceres subgroup','natal group',
        ]))
        return pd.Series({
            'is_karoo_supergroup': is_karoo,
            'is_cape_supergroup':  is_cape,
            'is_beaufort_group':   int(any(k in n for k in ['beaufort','tarkastad','adelaide','molteno','elliot','clarens','emakwezini'])),
            'is_ecca_group':       int(any(k in n for k in ['ecca','volksrust','vryheid','fort brown','waterford','prince albert','pietermaritzburg'])),
            'is_dwyka_group':      int('dwyka' in n),
            'is_karoo_dolerite':   int('dolerite' in n),
        })
    flags_df = df['macrostrat_name'].apply(flags)
    return pd.concat([df, flags_df], axis=1)

train = add_geology_flags(train)
val = add_geology_flags(val)

print("Geology flag coverage - train vs. val:")
for col in ['is_karoo_supergroup','is_cape_supergroup','is_beaufort_group',
            'is_ecca_group','is_dwyka_group','is_karoo_dolerite']:
    print(f"  {col:25s}: train={train[col].mean():.2f}  val={val[col].mean():.2f}")

Geology flag coverage - train vs. val:
  is_karoo_supergroup      : train=0.48  val=0.78
  is_cape_supergroup       : train=0.09  val=0.12
  is_beaufort_group        : train=0.16  val=0.61
  is_ecca_group            : train=0.19  val=0.01
  is_dwyka_group           : train=0.08  val=0.00
  is_karoo_dolerite        : train=0.04  val=0.16


In [25]:
from panel_model_v2 import target_encode_macrostrat

train, val = target_encode_macrostrat(train, val, TARGETS)
print('TE cols:', [c for c in train.columns if 'macrostrat_te' in c])

TE cols: ['macrostrat_te_Total Alkalinity', 'macrostrat_te_Electrical Conductance', 'macrostrat_te_Dissolved Reactive Phosphorus']


In [26]:
for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"  {target}")
    print('='*55)
    spatial_cv(train, target=target, n_splits=5)


  Total Alkalinity
  Fold 1: R²=0.4524  (n_test=1796)
  Fold 2: R²=0.2080  (n_test=1796)
  Fold 3: R²=0.2262  (n_test=1798)
  Fold 4: R²=0.3975  (n_test=1796)
  Fold 5: R²=0.3153  (n_test=1799)
  → Mean R² = 0.3199

  Electrical Conductance
  Fold 1: R²=0.3835  (n_test=1796)
  Fold 2: R²=0.3044  (n_test=1796)
  Fold 3: R²=0.4463  (n_test=1798)
  Fold 4: R²=0.4165  (n_test=1796)
  Fold 5: R²=0.2876  (n_test=1799)
  → Mean R² = 0.3677

  Dissolved Reactive Phosphorus
  Fold 1: R²=0.2831  (n_test=1796)
  Fold 2: R²=0.3025  (n_test=1796)
  Fold 3: R²=0.0243  (n_test=1798)
  Fold 4: R²=0.2295  (n_test=1796)
  Fold 5: R²=0.1842  (n_test=1799)
  → Mean R² = 0.2047


In [27]:
models = {}
for target in TARGETS:
    m = PanelWaterQualityModel(target=target)
    m.fit(train)
    models[target] = m
    print(f"Trained: {target}")

Trained: Total Alkalinity
Trained: Electrical Conductance
Trained: Dissolved Reactive Phosphorus


In [28]:
# Compute val site means from the val X features (no y used)
val_site_means = compute_site_means(val, TIME_VARIANT_COLS)

submission = submission_template.copy()
for target in TARGETS:
    preds = models[target].predict(val, site_means=val_site_means)
    submission[target] = np.clip(preds, 0, None)
    print(f"{target:40s}: mean={preds.mean():.1f}  min={preds.min():.1f}  max={preds.max():.1f}")

submission.to_csv("Data/submission.csv", index=False)
print("\nSaved → Data/submission.csv")
submission.head()

Total Alkalinity                        : mean=103.0  min=35.6  max=178.4
Electrical Conductance                  : mean=515.1  min=82.2  max=866.2
Dissolved Reactive Phosphorus           : mean=29.5  min=7.5  max=58.7

Saved → Data/submission.csv


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,103.404141,309.703251,12.061252
1,-33.329167,26.077500,16-09-2015,117.613330,583.804658,29.797444
2,-32.991639,27.640028,07-05-2015,103.317890,401.051968,30.675430
3,-34.096389,24.439167,07-02-2012,55.962354,515.283597,29.191484
4,-32.000556,28.581667,01-10-2014,95.728737,270.789235,16.094929


In [29]:
for target in TARGETS:
    score = spatial_cv(train, target=target, n_splits=5)
    print(f"{target:40s}: CV R² = {score:.3f}")

  Fold 1: R²=0.4524  (n_test=1796)
  Fold 2: R²=0.2080  (n_test=1796)
  Fold 3: R²=0.2262  (n_test=1798)
  Fold 4: R²=0.3975  (n_test=1796)
  Fold 5: R²=0.3153  (n_test=1799)
  → Mean R² = 0.3199
Total Alkalinity                        : CV R² = 0.320
  Fold 1: R²=0.3835  (n_test=1796)
  Fold 2: R²=0.3044  (n_test=1796)
  Fold 3: R²=0.4463  (n_test=1798)
  Fold 4: R²=0.4165  (n_test=1796)
  Fold 5: R²=0.2876  (n_test=1799)
  → Mean R² = 0.3677
Electrical Conductance                  : CV R² = 0.368
  Fold 1: R²=0.2831  (n_test=1796)
  Fold 2: R²=0.3025  (n_test=1796)
  Fold 3: R²=0.0243  (n_test=1798)
  Fold 4: R²=0.2295  (n_test=1796)
  Fold 5: R²=0.1842  (n_test=1799)
  → Mean R² = 0.2047
Dissolved Reactive Phosphorus           : CV R² = 0.205


In [30]:
spatial_cv(train, target='Dissolved Reactive Phosphorus', n_splits=5, between_weight=0.0)


  Fold 1: R²=-0.1131  (n_test=1796)
  Fold 2: R²=0.0640  (n_test=1796)
  Fold 3: R²=-0.0508  (n_test=1798)
  Fold 4: R²=-0.0154  (n_test=1796)
  Fold 5: R²=-0.1325  (n_test=1799)
  → Mean R² = -0.0496


np.float64(-0.0495730535971038)

In [31]:
pd.read_parquet("Extraction/soilgrids_cache.parquet").columns.tolist()


['Latitude',
 'Longitude',
 'sg_phh2o_0_5cm',
 'sg_phh2o_5_15cm',
 'sg_cec_0_5cm',
 'sg_cec_5_15cm',
 'sg_clay_0_5cm',
 'sg_clay_5_15cm',
 'sg_soc_0_5cm',
 'sg_soc_5_15cm']

In [32]:
[c for c in x_train_All.columns if c.startswith('sg_')]



[]

In [33]:
sub = pd.read_csv("Data/68f2201a56c154ae5e52b5b7-698be8ee8d80aea59d11f027-submission (1).csv")
comp = pd.read_csv("submission.csv")

In [34]:
sub == comp

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,True,True,True,False,False,False
1,True,True,True,False,False,False
2,True,True,True,False,False,False
3,True,True,True,False,False,False
4,True,True,True,False,False,False
...,...,...,...,...,...,...
195,True,True,True,False,False,False
196,True,True,True,False,False,False
197,True,True,True,False,False,False
198,True,True,True,False,False,False
